# មេរៀនទី 11 - Model Context Protocol (MCP)

The **Model Context Protocol (MCP)** គឺជា​ស្តង់ដារ​បើក​ដែលអនុញ្ញាតឲ្យភ្នាក់ងារ ស្វែងរក និងប្រើឧបករណ៍ ទ្រព្យសម្បត្តិ និងប្រភពទិន្នន័យ​នៅពេល​រត់កម្មវិធី។ ជំនួសការកូដឧបករណ៍តាំងក្នុងភ្នាក់ងារ MCP អនុញ្ញាតឲ្យភ្នាក់ងារ​តភ្ជាប់ទៅ​ម៉ាស៊ីនម៉ែម៉ែ внешний servers ដែលបង្ហាញសមត្ថភាព​តាមតម្រូវការ។

In this lesson, you'll learn:
- តើ MCP ជាអ្វី និងហេតុអ្វីវាសំខាន់សម្រាប់ប្រព័ន្ធភ្នាក់ងារ
- របៀបដែល​រចនាសម្ព័ន្ធ client-server នៃ MCP ធ្វើការ
- របៀបបង្កើតភ្នាក់ងារ​ដែលប្រើការស្វែងរកឧបករណ៍បែប MCP


## ការរៀបចំ

**ទាមទារមុន:**
- គម្រោង Azure AI Foundry ដែលមានម៉ូឌែលដែលបានដាក់ឲ្យដំណើរការ
- រត់ `az login` សម្រាប់ការផ្ទៀងផ្ទាត់អត្តសញ្ញាណ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## តើ Model Context Protocol (MCP) ជាអ្វី?

MCP កំណត់របៀបស្តង់ដារមួយសម្រាប់ភ្នាក់ងារ AI ដើម្បីស្វែងរក និងអន្តរកម្មជាមួយឧបករណ៍ និងប្រភពទិន្នន័យខាងក្រៅ៖

- **MCP Server**: បង្ហាញឧបករណ៍ ធនធាន និងសំណើតាមរយៈពិធីសាស្ត្រមានស្តង់ដារ
- **MCP Client**: វេលារត់របស់ភ្នាក់ងារ ដែលភ្ជាប់ទៅម៉ាស៊ីនបម្រើ និងស្វែងរកសមត្ថភាពដែលអាចប្រើបាន
- **Dynamic Discovery**: ភ្នាក់ងារមិនចាំបាច់មានឧបករណ៍ដែលបានកូដរឹង — ពួកវាស្វែងរកអ្វីដែលអាចប្រើបាននៅពេលរត់កម្មវិធី

នេះមានថាមពលសម្រាប់ការសាងសង់ប្រព័ន្ធភ្នាក់ងារដែលអាចពង្រីកបាន ដោយសមត្ថភាពថ្មីៗអាចត្រូវបានបន្ថែមដោយមិនចាំបាច់កែប្រែកូដភ្នាក់ងារ។


## របៀបដែល MCP ធ្វើការ

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ភ្នាក់ងារ (អតិថិជន MCP) តភ្ជាប់ទៅកាន់ម៉ាស៊ីនមេ MCP
2. ម៉ាស៊ីនមេឆ្លើយតបជាមួយបញ្ជីឧបករណ៍ដែលអាចប្រើបាន និងស្គីម៉ារបស់ពួកវា
3. ភ្នាក់ងារ អាចហៅឧបករណ៍ណាមួយដែលបានរកឃើញពេលវាកំពុងគិត
4. លទ្ធផលត្រឡប់មកវិញតាមពិធីសាស្ត្រដដែល


## ការសម្តែងការស្វែងរកឧបករណ៍ MCP

ដោយសារតែម៉ាស៊ីនមេ MCP ពិតប្រាកដត្រូវការកម្មវិធីម៉ាស៊ីនមេដែលកំពុងដំណើរការ យើងនឹងបង្ហាញគំរូនេះដោយប្រើមុខងារ `@tool` ដែលចម្លងអ្វីដែលសេវាស្នាក់នៅដែលភ្ជាប់ជាមួយ MCP នឹងផ្តល់។

នៅក្នុងបរិយាកាសផលិតកម្ម ឧបករណ៍ទាំងនេះនឹងត្រូវបានរកឃើញយ៉ាងឌីណាមិចពីម៉ាស៊ីនមេ MCP មិនមែនត្រូវបានកំណត់ផ្ទាល់ក្នុងម៉ាស៊ីនទេ។


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## កសាងភ្នាក់ងារមួយជាមួយឧបករណ៍បែប MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP ក្នុងបរិយាកាសផលិត

នៅក្នុងបរិយាកាសផលិត, MCP អនុញ្ញាតឱ្យមានលំនាំដែលមានសក្ដានុពលខ្លាំង:

- **ការរកឧបករណ៍ឌីណាមិច**: ភ្នាក់ងារតភ្ជាប់ទៅម៉ាស៊ីនមេ MCP ហើយរកឧបករណ៍នៅពេលដំណើរការ
- **ស្ថាបត្យកម្មដាច់ដោយឡែក**: អ្នកផ្គត់ផ្គង់ឧបករណ៍អាចធ្វើបច្ចុប្បន្នភាពដោយឯករាជ្យពីភ្នាក់ងារ
- **ការចែករំលែកអន្តអង្គភាព**: ក្រុមអាចបង្ហាញសមត្ថភាពតាមរយៈម៉ាស៊ីនមេ MCP ដែលភ្នាក់ងារណាមួយអាចប្រើបាន
- **ការគាំទ្រ Microsoft Agent Framework**: MAF មានការគាំទ្រអតិថិជន MCP ជាកំណត់ដើម តាមរយៈការរួមបញ្ចូល `mcp`

ដើម្បីប្រើម៉ាស៊ីនមេ MCP ពិតជាមួយ MAF អ្នកនឹងតភ្ជាប់តាមរយៈ `hosted_mcp_tool()` ឬការរួមបញ្ចូលអតិថិជន MCP។

**ស្វែងយល់បន្ថែម:**
- [សេចក្តីបញ្ជាក់ MCP](https://modelcontextprotocol.io/)
- [ការគាំទ្រ MCP របស់ Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## សេចក្តីសង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀន៖
- **MCP** គឺជា​ស្ដង់ដារបើក​សម្រាប់ការរកឧបករណ៍​ដោយឌីណាមិចរវាងភ្នាក់ងារ និងអ្នកផ្គត់ផ្គង់ឧបករណ៍
- **រចនាសម្ព័ន្ធ អតិថិជន-ម៉ាស៊ីនមេ** អនុញ្ញាតឲ្យភ្នាក់ងាររកឃើញសមត្ថភាពនៅពេលរត់
- MCP អនុញ្ញាតឲ្យ **ប្រព័ន្ធភ្នាក់ងារដែលអាចពង្រីកបាន និងបំបែកដោយឡែក** ដែលអាចបន្ថែមឧបករណ៍បានដោយមិនចាំបាច់ផ្លាស់ប្តូរកូដ
- Microsoft Agent Framework ផ្ដល់ **ការគាំទ្រ MCP បញ្ចូលក្នុងខ្លួន** សម្រាប់ការប្រើប្រាស់ក្នុងផលិតកម្ម


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator). ក្នុងខណៈដែលយើងខំប្រឹងសម្រាប់ភាពត្រឹមត្រូវ សូមចំណាំថាការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬការមិនត្រឹមត្រូវ។ ឯកសារដើមនៅក្នុងភាសាមូលដ្ឋានគួរត្រូវបានចាត់ទុកថាជាធនធានដែលមានសុពលភាព។ សម្រាប់ព័ត៌មានសំខាន់ យើងផ្តល់អនុសាសន៍ឱ្យធ្វើការបកប្រែដោយអ្នកបកប្រែជាមនុស្សដែលជាអ្នកជំនាញវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
